In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from category_encoders import TargetEncoder

import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cambiar ruta del dataset, ya que de forma parcial tira error

df = pd.read_csv(r"..\csv\cleaned_data.csv")
print("Forma del dataset:", df.shape)
print("\nPrimeras filas:")
df.head()

Forma del dataset: (50707, 17)

Primeras filas:


,Campaña,Pedido,Zona,Dirección entrega,Telefono,Barrio,Ciudad,Fecha recibo LD,Fecha reparto,Fecha entrega,Transportador,Estado,Fecha entrega corregida,Horas_entrega_corregida,Fecha reparto corregida,Horas_original,Horas_corregidas
0,202407,37087842,601,KR 47A 48 39- TEL 0-3143423923,0-3143423923,CORDOBA,CALI (VALLE),2024-04-27 19:07:31,2024-04-29 10:43:16,2024-04-29 22:08:43,CBZ922,A TIEMPO,2024-04-28 19:07:31,24.000000,2024-04-28 19:07:31,51.020000,24.000000
1,202407,37093297,580,CR 1H NORTE 77-10 SEGUNDO PISO- TEL 0-3154678944,0-3154678944,COMFENALCO,CALI (VALLE),2024-04-27 19:04:43,2024-04-28 10:24:04,2024-04-28 16:40:32,TZY274,A TIEMPO,2024-04-28 16:40:32,21.596944,2024-04-28 19:04:43,21.596944,21.596944
2,202407,37098642,594,CALLETERSERA SN124- TEL 0-3016931064,0-3016931064,PROVIVIENDA,DAGUA (VALLE),2024-04-28 20:40:40,2024-04-29 11:18:40,2024-04-30 12:00:25,TZN919,A TIEMPO,2024-04-30 12:00:25,39.329167,2024-04-30 20:40:40,39.329167,39.329167
3,202407,37105003,603,CL 18 A 9 A 11 PISO NO 1- TEL 0-3136142043,0-3136142043,CARLOS GUZMAN,PUERTO TEJADA (CAUCA),2024-05-04 09:36:18,2024-05-04 13:10:03,2024-05-05 23:00:44,CEN356,A TIEMPO,2024-05-05 23:00:44,37.407222,2024-05-06 09:36:18,37.407222,37.407222
4,202407,37111291,603,KR 18 15 A 40- TEL 0-3125418458,0-3125418458,LA SAMARIA,SANTANDER DE QUILICHAO (CAUCA),2024-05-04 09:18:31,2024-05-06 08:39:17,2024-05-06 22:00:29,TZY274,A TIEMPO,2024-05-06 09:18:31,48.000000,2024-05-06 09:18:31,60.699444,48.000000


In [3]:
# Seleccionar columnas relevantes
columnas_usar = ['Zona', 'Barrio', 'Ciudad', 'Fecha recibo LD', 'Fecha reparto', 
                 'Fecha entrega corregida', 'Transportador', 'Estado']

df_modelo = df[columnas_usar].copy()
print("Columnas a usar:", columnas_usar)
print("\nTipos de datos:")
print(df_modelo.dtypes)
print("\nValores únicos por columna:")
for col in df_modelo.columns:
    print(f"{col}: {df_modelo[col].nunique()} valores únicos")

Columnas a usar: ['Zona', 'Barrio', 'Ciudad', 'Fecha recibo LD', 'Fecha reparto', 'Fecha entrega corregida', 'Transportador', 'Estado']

Tipos de datos:
Zona                       str
Barrio                     str
Ciudad                     str
Fecha recibo LD            str
Fecha reparto              str
Fecha entrega corregida    str
Transportador              str
Estado                     str
dtype: object

Valores únicos por columna:
Zona: 9 valores únicos
Barrio: 734 valores únicos
Ciudad: 40 valores únicos
Fecha recibo LD: 44482 valores únicos
Fecha reparto: 50608 valores únicos
Fecha entrega corregida: 48541 valores únicos
Transportador: 26 valores únicos
Estado: 2 valores únicos


In [10]:
# Convertir fechas a datetime
df_modelo['Fecha recibo LD'] = pd.to_datetime(df_modelo['Fecha recibo LD'])

# Extraer características de fechas
df_modelo['Recibo_DayOfWeek'] = df_modelo['Fecha recibo LD'].dt.dayofweek
df_modelo['Recibo_Month'] = df_modelo['Fecha recibo LD'].dt.month
df_modelo['Recibo_Day'] = df_modelo['Fecha recibo LD'].dt.day

# Diferencias en horas
df_modelo['Horas_Recibo_Reparto'] = (df_modelo['Fecha reparto'] - df_modelo['Fecha recibo LD']).dt.total_seconds() / 3600
df_modelo['Horas_Reparto_Entrega'] = (df_modelo['Fecha entrega corregida'] - df_modelo['Fecha reparto']).dt.total_seconds() / 3600

In [11]:
# Codificar variable objetivo
le_estado = LabelEncoder()
y = le_estado.fit_transform(df_modelo['Estado'])
print("Clases codificadas:")
for clase, codigo in zip(le_estado.classes_, le_estado.transform(le_estado.classes_)):
    print(f"  {clase}: {codigo}")
print(f"\nDistribución de clases:")
print(pd.Series(y).value_counts())

Clases codificadas:
  A TIEMPO: 0
  TARDE: 1

Distribución de clases:
0    36158
1    14549
Name: count, dtype: int64


In [ ]:
# One-Hot Encoding para Transportador
transportador_encoded = pd.get_dummies(df_modelo['Transportador'], prefix='Transportador', drop_first=True)

# Target Encoding para Barrio
be = TargetEncoder(cols=['Barrio'])
barrio_encoded = be.fit_transform(df_modelo[['Barrio']], y)
barrio_encoded.columns = ['Barrio_TargetEncoded']

# Target Encoding para Ciudad
te = TargetEncoder(cols=['Ciudad'])
ciudad_encoded = te.fit_transform(df_modelo[['Ciudad']], y)
ciudad_encoded.columns = ['Ciudad_TargetEncoded']

print(f"Barrio - One-Hot: {barrio_encoded.shape[1]} features")
print(f"Transportador - One-Hot: {transportador_encoded.shape[1]} features")
print(f"Ciudad - Target Encoding: aplicado")

Barrio - One-Hot: 1 features
Transportador - One-Hot: 25 features
Ciudad - Target Encoding: aplicado


In [14]:
# Convertir Zona a numérica (Label Encoding)
le_zona = LabelEncoder()
zona_encoded = le_zona.fit_transform(df_modelo['Zona']).reshape(-1, 1)

# Construir matriz de features completa
X = pd.concat([
    pd.DataFrame(zona_encoded, columns=['Zona']),
    barrio_encoded,
    ciudad_encoded,
    transportador_encoded,
    df_modelo[['Recibo_DayOfWeek', 'Recibo_Month', 'Recibo_Day', 
               'Reparto_DayOfWeek', 'Reparto_Month', 
               'Horas_Recibo_Reparto', 'Horas_Reparto_Entrega']].reset_index(drop=True)
], axis=1)

print(f"Forma de X: {X.shape}")
print(f"Número total de features: {X.shape[1]}")
print(f"Primeras filas de X:")
print(X.head())

Forma de X: (50707, 35)
Número total de features: 35
Primeras filas de X:
   Zona  Barrio_TargetEncoded  Ciudad_TargetEncoded  Transportador_BDA446  \
0     5              0.302530              0.280612                 False   
1     0              0.252137              0.280612                 False   
2     3              0.254544              0.282895                 False   
3     6              0.302870              0.248000                 False   
4     6              0.151335              0.198327                 False   

   Transportador_CBL490  Transportador_CBZ922  Transportador_CEN356  \
0                 False                  True                 False   
1                 False                 False                 False   
2                 False                 False                 False   
3                 False                 False                  True   
4                 False                 False                 False   

   Transportador_CFG119  Transportad

In [15]:
# Normalizar features y dividir datos
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2, random_state=42, stratify=y)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (40565, 35)
X_test shape: (10142, 35)


In [16]:
# Entrenar modelo de regresión logística
logistic_reg = LogisticRegression(max_iter=1000, random_state=42)
logistic_reg.fit(X_train, y_train)

# Evaluar
train_accuracy = logistic_reg.score(X_train, y_train)
test_accuracy = logistic_reg.score(X_test, y_test)

print(f"Exactitud en entrenamiento: {train_accuracy:.4f}")
print(f"Exactitud en test: {test_accuracy:.4f}")

Exactitud en entrenamiento: 0.9702
Exactitud en test: 0.9700
